### Loading the env file

In [15]:
import os, pathlib, datetime as dt
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
RAW_DIR = pathlib.Path(os.path.join('..', os.getenv("DATA_DIR_RAW", "data/raw")))
PROC_DIR = pathlib.Path(os.path.join('..', os.getenv("DATA_DIR_PROCESSED", "data/processed")))
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
print("RAW_DIR:", RAW_DIR.resolve())
print("PROC_DIR:", PROC_DIR.resolve())

RAW_DIR: C:\Users\Munj Patel\Desktop\boot-4\bootcamp_munj_patel\homework\stage05_data-storage\data\raw
PROC_DIR: C:\Users\Munj Patel\Desktop\boot-4\bootcamp_munj_patel\homework\stage05_data-storage\data\processed


### Generating the data

In [16]:
import pandas as pd
import yfinance as yf
import numpy as np

In [17]:
ticker_data = yf.Ticker('^GSPC').history(period = 'max')

ticker_data['14_day_rolling_volatility'] = ticker_data['Close'].rolling(window = 14).std()
ticker_data['14_day_rolling_average'] = ticker_data['Close'].rolling(window = 14).mean()
ticker_data['14_day_rolling_median'] = ticker_data['Close'].rolling(window = 14).median()

ticker_data = ticker_data[['Close', '14_day_rolling_average', '14_day_rolling_median', '14_day_rolling_volatility']]

In [18]:
ticker_data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 24525 entries, 1927-12-30 00:00:00-05:00 to 2025-08-20 00:00:00-04:00
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Close                      24525 non-null  float64
 1   14_day_rolling_average     24512 non-null  float64
 2   14_day_rolling_median      24512 non-null  float64
 3   14_day_rolling_volatility  24512 non-null  float64
dtypes: float64(4)
memory usage: 958.0 KB


In [19]:
processed_file = ticker_data.dropna() # removes NaN values from the data

In [20]:
processed_file.columns

Index(['Close', '14_day_rolling_average', '14_day_rolling_median',
       '14_day_rolling_volatility'],
      dtype='object')

In [21]:
ticker_data.columns

Index(['Close', '14_day_rolling_average', '14_day_rolling_median',
       '14_day_rolling_volatility'],
      dtype='object')

### Saving to CSV (raw) and Parquet (processed)

In [9]:
!pip install fastparquet

   ---------------------------------------- 0.0/671.2 kB ? eta -:--:--
   ---------- ----------------------------- 184.3/671.2 kB 5.6 MB/s eta 0:00:01
   ---------------------------- ----------- 481.3/671.2 kB 6.0 MB/s eta 0:00:01
   ---------------------------------------  665.6/671.2 kB 6.0 MB/s eta 0:00:01
   ---------------------------------------- 671.2/671.2 kB 5.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB 8.9 MB/s eta 0:00:01
   -------------- ------------------------- 0.6/1.7 MB 7.6 MB/s eta 0:00:01
   ------------------------- -------------- 1.1/1.7 MB 8.5 MB/s eta 0:00:01
   ------------------------------ --------- 1.3/1.7 MB 7.4 MB/s eta 0:00:01
   ------------------------------------- -- 1.6/1.7 MB 7.2 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 6.4 MB/s eta 0:00:00


The value specified in an AutoRun registry key could not be parsed.
DEPRECATION: wfuzz 3.1.0 has a non-standard dependency specifier pyparsing>=2.4*; python_version >= "3.5". pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of wfuzz or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\Munj Patel\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [22]:
from datetime import datetime
from fastparquet import write
import os

In [23]:
def ts():
    return datetime.now().strftime('%Y%m%d-%H%M%S')

csv_path = os.path.join('..', 'data', 'raw', f"raw_prices_{ts()}.csv")
ticker_data.to_csv(csv_path, index=False)
print("Saved CSV →", csv_path)

parq_path = os.path.join('..', 'data', 'processed', f"processed_prices_{ts()}.parquet")
try:
    write(parq_path, processed_file)  # uses installed engine if available
    print("Saved Parquet →", parq_path)
except Exception as e:
    print("Parquet save failed (engine missing?). Skipping Parquet demo.")
    print("Error:", e)

Saved CSV → ..\data\raw\raw_prices_20250820-102441.csv
Saved Parquet → ..\data\processed\processed_prices_20250820-102441.parquet


### Reloading and Validating the data

In [24]:
def validate_loaded(original: pd.DataFrame, reloaded: pd.DataFrame, cols=('date','ticker','price')):
    checks = {
        'shape_equal': original.shape == reloaded.shape,
        'cols_present': all(c in reloaded.columns for c in cols)
    }
    # dtype sanity checks
    if 'price' in reloaded.columns:
        checks['price_is_numeric'] = pd.api.types.is_numeric_dtype(reloaded['price'])
    if 'date' in reloaded.columns:
        checks['date_is_datetime'] = pd.api.types.is_datetime64_any_dtype(reloaded['date'])
    return checks

raw_csv = pd.read_csv(csv_path)
print('CSV validation:', validate_loaded(ticker_data, raw_csv))

if os.path.exists(parq_path):
    try:
        processed_parq = pd.read_parquet(parq_path)
        print('Parquet validation:', validate_loaded(processed_file, processed_parq))
    except Exception as e:
        print('Parquet read failed:', e)
else:
    print('Parquet file not present (skipped earlier).')

CSV validation: {'shape_equal': True, 'cols_present': False}
Parquet validation: {'shape_equal': True, 'cols_present': False}


### Refractor to utilities

In [25]:
from typing import Union

def ensure_dir(path: pathlib.Path):
    path.parent.mkdir(parents=True, exist_ok=True)

def detect_format(path: Union[str, pathlib.Path]):
    suf = str(path).lower()
    if suf.endswith('.csv'): return 'csv'
    if suf.endswith('.parquet') or suf.endswith('.pq') or suf.endswith('.parq'): return 'parquet'
    raise ValueError('Unsupported format for: ' + str(path))

def write_ticker_data(ticker_data: pd.DataFrame, path: Union[str, pathlib.Path]):
    path = pathlib.Path(path)
    ensure_dir(path)
    fmt = detect_format(path)
    if fmt == 'csv':
        ticker_data.to_csv(path, index=False)
    elif fmt == 'parquet':
        try:
            ticker_data.to_parquet(path)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e
    return path

def read_ticker_data(path: Union[str, pathlib.Path]):
    path = pathlib.Path(path)
    fmt = detect_format(path)
    if fmt == 'csv':
        return pd.read_csv(path, parse_dates=['date']) if 'date' in pd.read_csv(path, nrows=0).columns else pd.read_csv(path)
    elif fmt == 'parquet':
        try:
            return pd.read_parquet(path)
        except Exception as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e

In [29]:
# utility usage
csv2 = RAW_DIR / f"prices_util_{ts()}.csv"
pq2  = PROC_DIR / f"prices_util_{ts()}.parquet"
write_ticker_data(ticker_data, csv2)
df2 = read_ticker_data(csv2)
print('Reloaded CSV via util, shape:', df2.shape)

Reloaded CSV via util, shape: (24525, 4)
